# Optional observation weights for Shapley attribution and model comparison

`ShapleyAttribution.fit` and `CompareAttribution.fit` take an optional
`weights=` argument: one nonnegative number per journey row that says how
much that row counts. Leaving it out fits every row with unit weight, which
is what every other example notebook does.

This notebook uses the same GA4 journey extract as the other examples to
show:

1. A fit without weights is unchanged.
2. Frequency weights: one row per distinct journey plus a count gives the
   same result as one row per journey.
3. Weights are kept as fractions, and a zero weight removes a journey.
4. A weighted comparison fits Markov and Shapley on the same weighted
   journeys.

In [1]:
# In Colab, install the package first:
# %pip install "do-attribution[viz,temporal]"
import textwrap

import numpy as np
import pandas as pd

from do_attribution import CompareAttribution, ShapleyAttribution

pd.set_option("display.width", 160)

WEIGHT_COLUMNS = ["weights_supplied", "total_weight", "weighted_conversions"]


def same(left, right):
    # Two mappings with the same keys and values equal to rtol=1e-10, atol=1e-12.
    keys = sorted(left)
    return keys == sorted(right) and bool(
        np.allclose(
            [left[key] for key in keys],
            [right[key] for key in keys],
            rtol=1e-10,
            atol=1e-12,
        )
    )

## Load the GA4 journeys

In [2]:
journeys = pd.read_csv("ga4_attribution_journeys.csv")

assert journeys["path"].notna().all() and (journeys["path"].str.len() > 0).all()
assert journeys["conversion"].isin([0, 1]).all()

paths = journeys["path"]
conversions = journeys["conversion"]
print(f"journeys   : {len(journeys):,}")
print(f"conversions: {conversions.sum():,}  (rate {conversions.mean():.4f})")

journeys   : 272,792
conversions: 4,848  (rate 0.0178)


## 1. Without weights, nothing changes

`weights` defaults to `None`. The call below is the one in
`basic_shapley.ipynb`, and it returns the same shares.

The `fit_diagnostics` table from `to_dataframes()` ends with three
observation-weight columns for every Shapley fit. For a fit without weights
they hold `False`, the number of journeys, and the number of conversions.

In [3]:
unweighted_model = ShapleyAttribution()
unweighted = unweighted_model.fit(paths, conversions)

shares = pd.Series(unweighted["attribution"], name="share").sort_values(ascending=False)
display(shares.round(4).to_frame())
display(unweighted_model.to_dataframes()["fit_diagnostics"][WEIGHT_COLUMNS])

,share
Referral,0.4307
Organic Search,0.3854
Direct,0.0986
Other,0.0668
Paid Search,0.0133
Email,0.0040
Organic Social,0.0009
Affiliate,0.0003


,weights_supplied,total_weight,weighted_conversions
0,False,272792.0,4848.0


## 2. Frequency weights

Most journeys in the extract repeat a path another journey already took.
Grouping identical `(path, conversion)` pairs gives one row per distinct
journey and a count of how many journeys it stands for.

In [4]:
frequency_table = (
    journeys.groupby(["path", "conversion"], as_index=False)
    .size()
    .rename(columns={"size": "journeys"})
)
print(f"{len(journeys):,} journey rows -> {len(frequency_table):,} distinct (path, conversion) rows")
display(frequency_table.sort_values("journeys", ascending=False).head())

272,792 journey rows -> 1,546 distinct (path, conversion) rows


,path,conversion,journeys
59,Direct,0,100623
393,Organic Search,0,73186
1224,Referral,0,37072
886,Other,0,20321
1109,Paid Search,0,5982


Passing the counts as `weights` fits that small table as if every journey
were listed. The same table without weights is a different, wrong dataset:
each distinct journey would count once.

In [5]:
frequency_model = ShapleyAttribution()
frequency_weighted = frequency_model.fit(
    frequency_table["path"],
    frequency_table["conversion"],
    weights=frequency_table["journeys"],
)
table_without_weights = ShapleyAttribution().fit(
    frequency_table["path"], frequency_table["conversion"]
)

display(
    pd.DataFrame(
        {
            "one row per journey": unweighted["attribution"],
            "frequency table, weighted": frequency_weighted["attribution"],
            "frequency table, no weights": table_without_weights["attribution"],
        }
    ).round(4)
)
print("weighted table reproduces the row-level shares         :",
      same(frequency_weighted["attribution"], unweighted["attribution"]))
print("weighted table reproduces the row-level raw values     :",
      same(frequency_weighted["shapley_values"], unweighted["shapley_values"]))
print("weighted table reproduces the row-level coalition table:",
      same(frequency_weighted["value_function"], unweighted["value_function"]))

,one row per journey,"frequency table, weighted","frequency table, no weights"
Affiliate,0.0003,0.0003,0.0048
Direct,0.0986,0.0986,0.2144
Email,0.0040,0.0040,0.0196
Organic Search,0.3854,0.3854,0.2409
Organic Social,0.0009,0.0009,0.0054
Other,0.0668,0.0668,0.1830
Paid Search,0.0133,0.0133,0.0525
Referral,0.4307,0.4307,0.2793


weighted table reproduces the row-level shares         : True
weighted table reproduces the row-level raw values     : True
weighted table reproduces the row-level coalition table: True


### What the summary and the export report

A weighted fit keeps two kinds of numbers apart:

- **Rows.** The `Data` table and `Path diagnostics` count the rows passed to
  `fit`, unweighted. Here that is the 1,546 rows of the frequency table, 276
  of them with `conversion = 1` — not journeys.
- **Weights.** The `Observation weights` section holds the total weight and
  the weighted conversions, which are the 272,792 journeys and 4,848
  conversions the table stands for. The coalition tables are labelled
  `Weighted conversions`.

The `fit_diagnostics` export carries the same weighted totals.

In [6]:
print(frequency_model.summary())

Attribution Summary
Model: ShapleyAttribution
Epoch source: static
Overall: 1546 paths, 276 conversions, 17.85% conversion rate

Data
+-----------+-------+-------------+---------+------------------------------------------------------+
| Cohort    | Paths | Conversions | Conv. % | Top channels                                         |
+-----------+-------+-------------+---------+------------------------------------------------------+
| All paths | 1546  | 276         | 17.85%  | Referral 43.07%, Organic Search 38.54%, Direct 9.86% |
+-----------+-------+-------------+---------+------------------------------------------------------+

Observation weights
+-------------------------------+--------+
| Metric                        | Value  |
+-------------------------------+--------+
| Total observation weight      | 272792 |
| Weighted conversions          | 4848   |
| Zero-weight journeys excluded | 0      |
+-------------------------------+--------+

Raw Shapley values (signed)
+---------

In [7]:
display(frequency_model.to_dataframes()["fit_diagnostics"][WEIGHT_COLUMNS])

,weights_supplied,total_weight,weighted_conversions
0,True,272792.0,4848.0


## 3. Fractional weights and zero weights

### Weights are kept as fractions

Weights do not have to be whole numbers. Dividing the counts by their total
turns them into relative frequencies that sum to 1. Rescaling every weight
by the same factor leaves the shares unchanged, and the coalition totals stay
fractional: nothing is rounded to an integer. With these weights the
weighted conversions are the observed conversion rate.

In [8]:
relative_frequency = frequency_table["journeys"] / frequency_table["journeys"].sum()

relative_model = ShapleyAttribution()
relative = relative_model.fit(
    frequency_table["path"], frequency_table["conversion"], weights=relative_frequency
)

print("shares unchanged by rescaling the weights:",
      same(relative["attribution"], unweighted["attribution"]))
relative_totals = relative_model.to_dataframes()["fit_diagnostics"][WEIGHT_COLUMNS]
display(relative_totals)
print(f"weighted conversions {relative_totals['weighted_conversions'].iloc[0]:.6f}; "
      f"observed conversion rate {conversions.mean():.6f}")

largest = sorted(relative["value_function"].items(), key=lambda item: -item[1])[:5]
display(
    pd.DataFrame(
        [
            {
                "coalition": " + ".join(coalition),
                "weighted conversions": value,
                "conversions, one row per journey": unweighted["value_function"][coalition],
            }
            for coalition, value in largest
        ]
    )
)

shares unchanged by rescaling the weights: True


,weights_supplied,total_weight,weighted_conversions
0,True,1.0,0.017772


weighted conversions 0.017772; observed conversion rate 0.017772


,coalition,weighted conversions,"conversions, one row per journey"
0,Referral,0.005073,1384
1,Organic Search,0.004744,1294
2,Organic Search + Referral,0.002452,669
3,Direct + Referral,0.001078,294
4,Direct + Organic Search + Referral,0.000675,184


### A zero weight removes a journey

A journey with zero weight is left out before the channel list and the
coalition table are built. A channel seen only on zero-weight journeys is
therefore absent from the result, rather than present with a zero share.

Below, every journey that touches `Affiliate` gets weight 0. The result is
the same as deleting those rows. The summary still counts all 272,792 rows
that were passed in, and reports how many had zero weight.

In [9]:
touches_affiliate = paths.str.split(" > ").apply(lambda touches: "Affiliate" in touches)
zero_weights = np.where(touches_affiliate, 0.0, 1.0)

zero_model = ShapleyAttribution()
zero_weighted = zero_model.fit(paths, conversions, weights=zero_weights)
rows_deleted = ShapleyAttribution().fit(
    paths[~touches_affiliate], conversions[~touches_affiliate]
)

print(f"journeys touching Affiliate (weight 0): {touches_affiliate.sum():,}")
print("channels without weights :", ", ".join(unweighted_model.get_channels()))
print("channels with zero weight:", ", ".join(zero_model.get_channels()))
print("same shares as deleting those rows:",
      same(zero_weighted["attribution"], rows_deleted["attribution"]))
print()
print(zero_model.summary().split("\n\nRaw Shapley values")[0])

journeys touching Affiliate (weight 0): 1,037
channels without weights : Affiliate, Direct, Email, Organic Search, Organic Social, Other, Paid Search, Referral
channels with zero weight: Direct, Email, Organic Search, Organic Social, Other, Paid Search, Referral
same shares as deleting those rows: True

Attribution Summary
Model: ShapleyAttribution
Epoch source: static
Overall: 272792 paths, 4848 conversions, 1.78% conversion rate

Data
+-----------+--------+-------------+---------+------------------------------------------------------+
| Cohort    | Paths  | Conversions | Conv. % | Top channels                                         |
+-----------+--------+-------------+---------+------------------------------------------------------+
| All paths | 272792 | 4848        | 1.78%   | Referral 43.08%, Organic Search 38.55%, Direct 9.86% |
+-----------+--------+-------------+---------+------------------------------------------------------+

Observation weights
+---------------------------

## 4. Weighted comparison

`CompareAttribution.fit(..., weights=)` drops zero-weight journeys once and
passes the remaining journeys and weights to both models, so the Markov and
Shapley columns always describe the same weighted journeys.

A common reason to need this is a down-sampled table. Suppose the journey
table keeps every converting journey and every multi-touch journey, but only
two of every five single-touch journeys that did not convert. Each of those
kept rows then stands for 2.5 journeys.

In [10]:
single_touch_non_converting = (paths.str.count(" > ") == 0) & (conversions == 0)
kept = ~single_touch_non_converting | (np.arange(len(journeys)) % 5 < 2)

sample = journeys[kept]
sample_weights = np.where(single_touch_non_converting[kept], 2.5, 1.0)

print(f"rows kept           : {len(sample):,} of {len(journeys):,}")
print(f"total weight        : {sample_weights.sum():,.1f}")
print(f"weighted conversions: {(sample_weights * sample['conversion']).sum():,.1f}")

rows kept           : 129,749 of 272,792
total weight        : 273,164.0
weighted conversions: 4,848.0


In [11]:
def compare(paths, conversions, weights=None):
    comparison = CompareAttribution(markov_removal_strategy="redirect_to_null")
    comparison.fit(paths, conversions, weights=weights)
    return comparison


comparisons = {
    "all journeys": compare(paths, conversions),
    "sample, weighted": compare(sample["path"], sample["conversion"], sample_weights),
    "sample, no weights": compare(sample["path"], sample["conversion"]),
}
frames = {name: comparison.attribution_frame() for name, comparison in comparisons.items()}
display(pd.concat(frames, axis=1).round(4))

all journeys         sample, weighted         sample, no weights        
                     markov shapley           markov shapley             markov shapley
Affiliate            0.0019  0.0003           0.0019  0.0003             0.0030  0.0003
Direct               0.1077  0.0986           0.1075  0.0986             0.1332  0.0986
Email                0.0040  0.0040           0.0040  0.0040             0.0037  0.0040
Organic Search       0.3621  0.3854           0.3621  0.3854             0.3603  0.3854
Organic Social       0.0012  0.0009           0.0012  0.0009             0.0015  0.0009
Other                0.0801  0.0668           0.0800  0.0668             0.0895  0.0668
Paid Search          0.0140  0.0133           0.0140  0.0133             0.0168  0.0133
Referral             0.4290  0.4307           0.4292  0.4307             0.3921  0.4307

In [12]:
reference = frames["all journeys"]
largest_difference = pd.DataFrame(
    {
        name: (frame - reference).abs().max()
        for name, frame in frames.items()
        if name != "all journeys"
    }
).T
print("Largest share difference from the fit on all journeys:")
display(largest_difference.round(4))

for name, comparison in comparisons.items():
    probability = comparison.get_results()["markov"]["total_conversion_probability"]
    print(f"{name:<18}: Markov P(conversion | start) = {probability:.4f}")

Largest share difference from the fit on all journeys:


,markov,shapley
"sample, weighted",0.0002,0.0
"sample, no weights",0.0369,0.0


all journeys      : Markov P(conversion | start) = 0.0178
sample, weighted  : Markov P(conversion | start) = 0.0177
sample, no weights: Markov P(conversion | start) = 0.0374


With the weights, the down-sampled table gives Markov shares within 0.0002
of the fit on all journeys, and a conversion probability of 0.0177 against
0.0178. Without them, the sample over-represents converting journeys: the
conversion probability roughly doubles to 0.0374 and Markov shares move by
up to 3.7 percentage points.

The Shapley column is identical in all three fits. Its coalition table
counts only converting journeys, and this sample keeps every one of them
with weight 1, so the weights on non-converting journeys do not enter it.
Markov models the non-converting journeys too, so it needs the weights.

### Baselines and extra models are not weighted

The built-in first-touch and last-touch baselines and any `extra_models` do
not accept weights. A weighted comparison that includes them is refused
before any model is fitted, so a weighted column is never placed beside an
unweighted one.

In [13]:
try:
    CompareAttribution(include_baselines=True).fit(
        sample["path"], sample["conversion"], weights=sample_weights
    )
except ValueError as error:
    print(textwrap.fill(f"ValueError: {error}", width=88))

ValueError: A weighted comparison supports only the built-in Markov and Shapley models.
The baselines and extra_models do not accept observation weights and would be fitted
unweighted beside them: ['first_touch', 'last_touch']. Construct CompareAttribution with
include_baselines=False and without extra_models, or call fit() without weights.


## Weight rules

- One weight per row, matched by position. The index labels of a pandas
  Series are ignored.
- Weights must be finite and nonnegative, and their total must be positive.
  Booleans count as 0 and 1.
- Paths and outcomes are validated on every row, including rows with zero
  weight, before those rows are dropped.
- If no journey with positive weight converts, `attribution` is `None` with
  `normalization_reason="zero_raw_magnitude"`.
- `MarkovAttribution.fit` takes `weights` under the same rules. On its own
  it keeps the channels of zero-weight journeys in its channel list;
  `CompareAttribution` drops those journeys for both models.
- The temporal models and the baselines do not take weights.